In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))
ART = ROOT / "artifacts"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from plotting import series_styles, new_figure

plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

def es(v, d=4):
    """Numero con formato de Colombia: punto de miles y coma decimal."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}".replace(",", ".")
    s = f"{float(v):,.{d}f}"
    return s.replace(",", "X").replace(".", ",").replace("X", ".")

def tabla(df, d=4, enteros=()):
    out = df.copy()
    for c in out.columns:
        if c in enteros:
            out[c] = out[c].map(lambda v: es(int(v)) if pd.notna(v) else "")
        elif pd.api.types.is_float_dtype(out[c]):
            out[c] = out[c].map(lambda v: es(v, d))
        elif pd.api.types.is_integer_dtype(out[c]):
            out[c] = out[c].map(es)
    display(out.style.hide(axis="index"))

def figura(ruta, ancho=None):
    display(Image(filename=str(ruta), width=ancho))

# 1. Ficha de datos y preguntas de investigación

Este capítulo fija el punto de partida del proyecto: qué datos se usan, de dónde vienen, qué tamaño tienen y qué limitaciones arrastran, y qué preguntas debe responder el trabajo. Esto es primordial porque todas las decisiones posteriores, desde la partición temporal hasta el tamaño de las muestras, se derivan de la ficha y de las hipótesis declaradas aquí, antes de ver un solo resultado.

## 1.1 Objetivo y alcance

El objetivo es construir, evaluar y comparar clasificadores MLP para predecir si un usuario hace clic en un anuncio móvil (`click = 1`) o no (`click = 0`), en scikit-learn y en PySpark, bajo un protocolo reproducible y libre de fuga de información, y analizar de forma crítica la interpretabilidad local con LIME contrastada con SHAP {cite}`enunciado212`.

El enunciado recomienda Criteo 1TB como dataset principal, pero contempla de forma explícita una alternativa sin clúster: usar Avazu completo y tratar la comparación entre entornos como un experimento de punto de cruce, sin presuponer que Spark ganará. Por tal motivo, se opta por esa ruta, con tres condiciones de alcance declaradas: Avazu completo, PySpark en modo local (`local[*]`, sin clúster real) y grillas de hiperparámetros reducidas allí donde el tiempo de cómputo medido lo exigió, siempre dentro del espacio de valores que fija el enunciado.

## 1.2 Ficha de datos

La ficha reúne los campos que el enunciado declara obligatorios: origen, licencia, período, tasa base de clics, tamaño en disco y en memoria, y limitaciones. El tamaño en memoria se midió sobre el día 30 completo (4.218.938 filas) y se extrapoló linealmente al total de filas, porque cargar el dataset entero en pandas no es posible en esta máquina.

In [2]:
ds = json.loads((ART / "eda" / "data_sheet.json").read_text(encoding="utf-8"))
ficha = pd.DataFrame([
    ("Origen", "Kaggle, competencia Avazu Click-Through Rate Prediction (2014)"),
    ("Licencia", "Uso sujeto a las reglas de la competencia, aceptadas en Kaggle para la descarga"),
    ("Período", "21 al 30 de octubre de 2014 (10 días; `hour` en formato YYMMDDHH)"),
    ("Filas", es(ds["n_filas"])),
    ("Tasa base de clics", es(100 * ds["tasa_base_click"], 2) + " %"),
    ("Tamaño en disco (CSV)", es(ds["tamano_csv_crudo_gb"], 2) + " GB"),
    ("Tamaño en disco (Parquet por día)", es(ds["tamano_parquet_gb"], 2) + " GB"),
    ("Tamaño en memoria, Arrow (estimado)", "10,11 GB"),
    ("Tamaño en memoria, pandas (estimado)", "30,08 GB"),
    ("Partición temporal", "train días 21 a 28, validación día 29, prueba día 30"),
    ("Hardware", "Intel Core i5-1235U (10 núcleos, 12 hilos), 16 GB de RAM, sin clúster"),
], columns=["Campo", "Valor"])
display(ficha.style.hide(axis="index"))

Campo,Valor
Origen,"Kaggle, competencia Avazu Click-Through Rate Prediction (2014)"
Licencia,"Uso sujeto a las reglas de la competencia, aceptadas en Kaggle para la descarga"
Período,21 al 30 de octubre de 2014 (10 días; `hour` en formato YYMMDDHH)
Filas,40.428.967
Tasa base de clics,"16,98 %"
Tamaño en disco (CSV),"5,88 GB"
Tamaño en disco (Parquet por día),"1,63 GB"
"Tamaño en memoria, Arrow (estimado)","10,11 GB"
"Tamaño en memoria, pandas (estimado)","30,08 GB"
Partición temporal,"train días 21 a 28, validación día 29, prueba día 30"


**Interpretación.** Avazu tiene 40.428.967 impresiones, de las cuales el 16,98 % terminó en clic, lo que equivale a una razón de 4,89 negativos por cada positivo. El CSV ocupa 5,88 GB y el Parquet particionado por día 1,63 GB, una compresión de 3,6 veces, propia del formato columnar con codificación por diccionario y compresión por bloques. En memoria, el mismo volumen ocupa unos 10,1 GB en Arrow y unos 30 GB como DataFrame de pandas, porque cada cadena de texto pasa a ser un objeto de Python con su propia cabecera.

Hay que tener en cuenta que los 30 GB de pandas son casi el doble de la RAM disponible, y que el enunciado ya advierte que Avazu cabe en una máquina de 32 a 64 GB. El desbalance moderado no impide usar ROC AUC como métrica primaria, porque es invariante a la tasa base, pero sí obliga a reportar PR AUC y a elegir el umbral de decisión en validación, ya que un umbral de 0,5 clasificaría casi todo como no clic.

Por tal motivo, se opta por leer siempre desde Parquet con DuckDB y pyarrow, fuera de memoria, y por trabajar con muestras escalonadas en scikit-learn. Spark lee los mismos archivos, y el tamaño efectivo de cada muestra queda documentado en el capítulo 3.

## 1.3 Diccionario de variables

Las variables de Avazu son categóricas casi en su totalidad, y buena parte están anonimizadas: los identificadores vienen como códigos hexadecimales y `C1` y `C14` a `C21` no tienen descripción pública. La tabla separa las variables originales de las derivadas del campo `hour`.

In [3]:
card = pd.read_csv(ART / "eda" / "cardinalidad_resumen.csv").set_index("columna")["n_categorias"]
dic = pd.DataFrame([
    ("click", "objetivo", "1 si la impresión terminó en clic, 0 si no"),
    ("hour", "temporal", "fecha y hora de la impresión (YYMMDDHH)"),
    ("C1", "categórica anonimizada", "sin descripción pública"),
    ("banner_pos", "categórica", "posición del anuncio en la pantalla"),
    ("site_id, site_domain, site_category", "categóricas", "sitio web donde se mostró el anuncio"),
    ("app_id, app_domain, app_category", "categóricas", "aplicación donde se mostró el anuncio"),
    ("device_id, device_ip, device_model", "categóricas de alta cardinalidad", "identificadores del dispositivo"),
    ("device_type, device_conn_type", "categóricas", "tipo de dispositivo y de conexión"),
    ("C14 a C21", "categóricas anonimizadas", "códigos numéricos sin descripción pública"),
    ("day, hour_of_day, franja_horaria", "derivadas", "día, hora y franja (madrugada, mañana, tarde, noche) desde `hour`"),
], columns=["Variable", "Tipo", "Descripción"])
display(dic.style.hide(axis="index"))
print("Categorías distintas en las variables de alta cardinalidad:")
print(card.map(es).to_string())

Variable,Tipo,Descripción
click,objetivo,"1 si la impresión terminó en clic, 0 si no"
hour,temporal,fecha y hora de la impresión (YYMMDDHH)
C1,categórica anonimizada,sin descripción pública
banner_pos,categórica,posición del anuncio en la pantalla
"site_id, site_domain, site_category",categóricas,sitio web donde se mostró el anuncio
"app_id, app_domain, app_category",categóricas,aplicación donde se mostró el anuncio
"device_id, device_ip, device_model",categóricas de alta cardinalidad,identificadores del dispositivo
"device_type, device_conn_type",categóricas,tipo de dispositivo y de conexión
C14 a C21,categóricas anonimizadas,códigos numéricos sin descripción pública
"day, hour_of_day, franja_horaria",derivadas,"día, hora y franja (madrugada, mañana, tarde, noche) desde `hour`"


Categorías distintas en las variables de alta cardinalidad:
columna
site_id             4.737
site_domain         7.745
app_id              8.552
app_domain            559
device_id       2.686.408
device_ip       6.729.486
device_model        8.251


## 1.4 Preguntas de investigación e hipótesis

El enunciado plantea cuatro preguntas y dos hipótesis explícitas {cite}`enunciado212`. Se reproducen tal como aparecen, junto con el capítulo donde se responde cada una.

| Pregunta o hipótesis | Se responde en |
|---|---|
| **P1.** ¿A partir de qué volumen PySpark supera a scikit-learn en tiempo de entrenamiento sin degradar la calidad predictiva? | Capítulos 7 y 10 |
| **H1.** Existe un punto de cruce medible, y por debajo de él scikit-learn es más eficiente. | Capítulo 7 |
| **P2.** ¿Cómo afecta la codificación de variables de alta cardinalidad (hashing, frecuencia, target encoding) al desempeño y al costo computacional? | Capítulo 9 |
| **H2.** El hashing con dimensión moderada es comparable a la codificación por frecuencia con una fracción de la memoria. | Capítulo 9 |
| **P3.** ¿Aporta el MLP una mejora estadísticamente significativa sobre líneas base más simples (regresión logística, árboles potenciados)? | Capítulo 6 |
| **P4.** ¿Son las explicaciones de LIME estables y fieles al modelo? | Capítulo 8 |

## 1.5 Decisiones fijadas antes de entrenar

Las decisiones de la tabla se fijaron antes de ver los resultados y se heredan en todo el proyecto. Fijarlas a priori evita ajustar la métrica, el margen de equivalencia o la partición a lo que convenga después, que es una forma silenciosa de sobreajuste del análisis.

| Decisión | Valor |
|---|---|
| Tipo de problema | Clasificación binaria supervisada |
| Métrica primaria | ROC AUC |
| Métricas secundarias | PR AUC, log loss, Brier, ECE, y F1, precision y recall con umbral elegido en validación |
| Partición | Temporal: train días 21 a 28, validación día 29, prueba día 30 |
| Unidad de independencia | La semilla de entrenamiento (5 semillas: 13, 29, 47, 71 y 97) |
| Nivel de significancia | $\alpha = 0{,}05$, con corrección de Holm dentro de cada familia de pruebas |
| Margen de equivalencia | $\delta = 0{,}01$ en ROC AUC, para la prueba TOST |
| Umbral de decisión | El que maximiza F1 en validación, congelado antes de evaluar en prueba |

El margen $\delta = 0{,}01$ quiere decir que dos modelos cuya diferencia de AUC cae dentro de un punto porcentual se consideran equivalentes en la práctica. No proviene de una norma externa: es una convención fijada antes de ver resultados, para que la conclusión de equivalencia no dependa de ellos. Por eso el capítulo 6 reporta, además de la conclusión, la diferencia exacta con su intervalo de confianza, de forma que el lector pueda aplicar un margen distinto si su contexto lo exige.